# Using a Variational Autoencoder for Compression and Denoising

This notebook trains a VAE on MNIST and uses it for two tasks:
- Data compression
- Image denoising

It trains three VAEs with latent dimensions 2, 16 and 32 so the results can be compared.

## Step 1: Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 2: Load the MNIST dataset

In [ ]:
train_data = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST("data", train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)

print("Training images:", len(train_data))
print("Test images:", len(test_data))

## Step 3: Build the VAE

- The **encoder** turns a 784-pixel image into two small vectors: `mu` (mean) and `logvar` (log variance).
- The **latent vector `z`** is picked using `mu` and `logvar`. This is the compressed form of the image.
- The **decoder** turns `z` back into a 784-pixel image.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.hidden = nn.Linear(784, 400)
        self.to_mu = nn.Linear(400, latent_dim)
        self.to_logvar = nn.Linear(400, latent_dim)
        self.from_latent = nn.Linear(latent_dim, 400)
        self.output = nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.to_mu(h), self.to_logvar(h)

    def sample(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        noise = torch.randn_like(std)
        return mu + noise * std

    def decode(self, z):
        h = F.relu(self.from_latent(z))
        return torch.sigmoid(self.output(h))

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.sample(mu, logvar)
        return self.decode(z), mu, logvar

## Step 4: Loss function and training

The VAE loss has two parts:
- **Reconstruction loss**: how different the output is from the input
- **KL divergence**: keeps the latent space smooth and well organised

In [ ]:
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon, x.view(-1, 784), reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss


def train_vae(latent_dim, epochs=10):
    model = VAE(latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        total_loss = 0
        for images, _ in train_loader:
            images = images.to(device)
            recon, mu, logvar = model(images)
            loss = vae_loss(recon, images, mu, logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Latent {latent_dim} | Epoch {epoch + 1}/{epochs} | Loss per image: {total_loss / len(train_data):.2f}")

    return model

## Step 5: Train VAEs with latent dimensions 2, 16 and 32

In [ ]:
latent_dims = [2, 16, 32]
models = {}

for d in latent_dims:
    models[d] = train_vae(d)
    print()

## Part A: Compression

Each MNIST image has 784 values. The latent vector has only `latent_dim` values.

Compression ratio = 784 / latent dimension

In [ ]:
original_size = 784
compression_ratios = {}

for d in latent_dims:
    compression_ratios[d] = original_size / d
    print(f"Latent dimension {d}: {original_size} values -> {d} values | Compression ratio = {compression_ratios[d]:.1f}x")

## Part B: Reconstruction

We take the first 10 test images, encode them, decode them and calculate the Mean Squared Error (MSE) for each image.

In [ ]:
test_images = torch.stack([test_data[i][0] for i in range(10)]).to(device)


def reconstruct(model, images):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(images.view(-1, 784))
        output = model.decode(mu)
    return output.view(-1, 1, 28, 28)


def mse_per_image(a, b):
    return ((a - b) ** 2).mean(dim=(1, 2, 3))


def show_images(rows, row_names, title):
    fig, axes = plt.subplots(len(rows), 10, figsize=(15, 1.7 * len(rows)))
    for r, images in enumerate(rows):
        for i in range(10):
            axes[r, i].imshow(images[i].cpu().squeeze(), cmap="gray")
            axes[r, i].axis("off")
        axes[r, 0].set_title(row_names[r], fontsize=9, loc="left")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
reconstructions = {}
recon_mse = {}

for d in latent_dims:
    reconstructions[d] = reconstruct(models[d], test_images)
    recon_mse[d] = mse_per_image(reconstructions[d], test_images)

    print(f"Latent dimension {d}")
    for i in range(10):
        print(f"  Image {i + 1}: MSE = {recon_mse[d][i].item():.5f}")
    print(f"  Average MSE = {recon_mse[d].mean().item():.5f}")
    print()

In [ ]:
for d in latent_dims:
    show_images([test_images, reconstructions[d]], ["Original", "Reconstructed"], f"Reconstruction with latent dimension {d}")

## Part C: Denoising

We add random Gaussian noise to the 10 test images and pass them through each trained VAE.
The VAE has only learned to draw clean digits, so it tends to remove the noise.

In [ ]:
torch.manual_seed(0)

noise_level = 0.3
noisy_images = test_images + noise_level * torch.randn_like(test_images)
noisy_images = noisy_images.clamp(0, 1)

noisy_mse = mse_per_image(noisy_images, test_images).mean().item()
print(f"Average MSE of noisy images vs clean images: {noisy_mse:.5f}")

In [ ]:
denoised = {}
denoised_mse = {}

for d in latent_dims:
    denoised[d] = reconstruct(models[d], noisy_images)
    denoised_mse[d] = mse_per_image(denoised[d], test_images).mean().item()
    print(f"Latent dimension {d}: denoised MSE vs clean image = {denoised_mse[d]:.5f}")

In [ ]:
for d in latent_dims:
    show_images([test_images, noisy_images, denoised[d]], ["Clean", "Noisy", "Denoised"], f"Denoising with latent dimension {d}")

## Part D: Changing the latent dimension

Below we compare the three models side by side on the same 10 test images.

In [ ]:
show_images(
    [test_images] + [reconstructions[d] for d in latent_dims],
    ["Original"] + [f"Latent {d}" for d in latent_dims],
    "Reconstruction comparison for different latent dimensions"
)

In [ ]:
show_images(
    [test_images, noisy_images] + [denoised[d] for d in latent_dims],
    ["Clean", "Noisy"] + [f"Denoised (latent {d})" for d in latent_dims],
    "Denoising comparison for different latent dimensions"
)

## Results table

In [ ]:
def quality_label(mse):
    if mse < 0.01:
        return "Very good (sharp)"
    elif mse < 0.03:
        return "Good"
    else:
        return "Poor (blurry)"


def denoising_note(clean_mse):
    if clean_mse < noisy_mse:
        return f"Noise reduced (MSE {noisy_mse:.4f} -> {clean_mse:.4f})"
    else:
        return f"Noise not reduced (MSE {noisy_mse:.4f} -> {clean_mse:.4f})"


rows = []
for d in latent_dims:
    avg_mse = recon_mse[d].mean().item()
    rows.append({
        "Latent dimension": d,
        "Compression ratio": f"{compression_ratios[d]:.1f}x",
        "Reconstruction error (MSE)": round(avg_mse, 5),
        "Reconstruction quality": quality_label(avg_mse),
        "Denoising observations": denoising_note(denoised_mse[d])
    })

results = pd.DataFrame(rows)
results

## Conclusion

- **Compression:** A smaller latent dimension gives a higher compression ratio. Latent dimension 2 compresses 784 values into 2 (392x), but loses a lot of detail.
- **Reconstruction:** Reconstruction error goes down and images become sharper as the latent dimension increases from 2 to 16 to 32.
- **Denoising:** The VAE removes most of the noise because it only knows how to generate clean digit-like images. With a very small latent space the output looks like a different digit sometimes, and with a larger latent space it keeps more detail from the original.
- **Trade-off:** More latent dimensions give better quality but less compression.